In [2]:
!pip install pandas numpy yfinance ta matplotlib

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt

data = yf.download("AAPL", period="1mo", interval="1d")
data.tail()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2026-06-15,296.420013,297.779999,291.700012,294.119995,45732600
2026-06-16,299.239990,300.480011,293.970001,295.250000,39874400
2026-06-17,295.950012,302.070007,294.359985,300.850006,42745100
2026-06-18,298.010010,300.570007,295.619995,298.109985,85962200
2026-06-22,297.010010,302.420013,296.760010,297.500000,40202659


In [5]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import matplotlib.pyplot as plt

symbol = "AAPL"

daily = yf.download(symbol, period="2y", interval="1d")
hourly = yf.download(symbol, period="730d", interval="1h")
two_min = yf.download(symbol, period="60d", interval="2m")

daily.tail()

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2026-06-15,296.420013,297.779999,291.700012,294.119995,45732600
2026-06-16,299.239990,300.480011,293.970001,295.250000,39874400
2026-06-17,295.950012,302.070007,294.359985,300.850006,42745100
2026-06-18,298.010010,300.570007,295.619995,298.109985,85962200
2026-06-22,297.010010,302.420013,296.760010,297.500000,40202659


In [7]:
daily.columns

MultiIndex([( 'Close', 'AAPL'),
            (  'High', 'AAPL'),
            (   'Low', 'AAPL'),
            (  'Open', 'AAPL'),
            ('Volume', 'AAPL'),
            (  'EMA9',     ''),
            ( 'EMA20',     ''),
            ( 'EMA50',     '')],
           names=['Price', 'Ticker'])

In [8]:
daily = yf.download("AAPL", period="2y", interval="1d", auto_adjust=False)
hourly = yf.download("AAPL", period="730d", interval="1h", auto_adjust=False)
two_min = yf.download("AAPL", period="60d", interval="2m", auto_adjust=False)

daily.columns = daily.columns.get_level_values(0)
hourly.columns = hourly.columns.get_level_values(0)
two_min.columns = two_min.columns.get_level_values(0)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [9]:
def add_emas(df):
    df["EMA9"] = df["Close"].ewm(span=9).mean()
    df["EMA20"] = df["Close"].ewm(span=20).mean()
    df["EMA50"] = df["Close"].ewm(span=50).mean()
    return df

daily = add_emas(daily)
hourly = add_emas(hourly)
two_min = add_emas(two_min)

daily["ATR"] = ta.volatility.average_true_range(
    high=daily["High"],
    low=daily["Low"],
    close=daily["Close"],
    window=14
)

daily.tail()

Price,Adj Close,Close,High,Low,Open,Volume,EMA9,EMA20,EMA50,ATR
Date,,,,,,,,,,
2026-06-15,296.420013,296.420013,297.779999,291.700012,294.119995,45732600,298.091143,298.905457,289.443341,7.272720
2026-06-16,299.239990,299.239990,300.480011,293.970001,295.250000,39874400,298.320913,298.937317,289.827524,7.218241
2026-06-17,295.950012,295.950012,302.070007,294.359985,300.850006,42745100,297.846733,298.652812,290.067621,7.253368
2026-06-18,298.010010,298.010010,300.570007,295.619995,298.109985,85962200,297.879388,298.591593,290.379088,7.088843
2026-06-22,297.010010,297.010010,302.420013,296.760010,297.500000,40202659,297.705512,298.440966,290.639124,6.986783


In [11]:
weekly = daily.resample("W").agg({
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum"
}).dropna()

monthly = daily.resample("ME").agg({
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum"
}).dropna()

weekly = add_emas(weekly)
monthly = add_emas(monthly)

In [12]:
monthly.tail()

Price,Open,High,Low,Close,Volume,EMA9,EMA20,EMA50
Date,,,,,,,,
2026-02-28,260.029999,280.910004,255.449997,264.179993,988921000,254.484226,245.070729,239.773790
2026-03-31,262.410004,266.529999,245.509995,253.789993,900035700,254.344347,246.004399,240.712945
2026-04-30,254.080002,276.000000,245.699997,271.350006,907538500,257.765675,248.686672,242.710275
2026-05-31,278.859985,315.000000,274.859985,312.059998,981286400,268.676062,255.323062,247.116934
2026-06-30,309.630005,317.399994,287.380005,297.010010,791083659,274.364342,259.647477,250.211976


In [13]:
last_month = monthly.iloc[-1]

monthly_pass = (
    last_month["Close"] > last_month["EMA9"]
    and last_month["Close"] > last_month["EMA20"]
    and last_month["Close"] > last_month["EMA50"]
)

print(monthly_pass)

True
